# 17 · eval — **baseline** (+ `act_te`) / transfer

5모델 × 4 seed × 5 rep = **100 run**. `act_te` 는 act 체크포인트 + eval-time TE.

**150k 체크포인트 × 5회 반복 × 500 에피소드** (rep 마다 `--seed = 1000 + 100·rep` → env 초기상태 변경).
insertion 과 같은 프로토콜이라 두 task 를 그대로 비교할 수 있다. 끝난 run 은 자동 skip.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.SHORT_SIM       # ★ 'transfer' (AlohaTransferCube-v0) — 짧은 앵커
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3]
GPUS  = cf.v23.available_gpus()
NGPU  = len(GPUS)
TAGS = cf.GROUP_BASELINE + ['act_te']
REPS = list(range(cf.EVAL_REPEATS))
N_EP = cf.EVAL_N_EP

print('GPU :', GPUS, f'({NGPU}개)')
print('task:', TASK, '| ckpt', f'{cf.CKPT_STEP:,}', '| reps', REPS, '| n_ep', N_EP)
print('eval:', TAGS, '| 총 run:', len(TAGS) * len(SEEDS) * len(REPS))

## 사전 확인 — 150k 체크포인트 (transfer)

In [ ]:
ok = cf.print_ckpt_status(cf.GROUP_BASELINE, SEEDS, TASK)

## 반복 eval

In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, ngpu=NGPU, n_episodes=N_EP)

## 결과 (SR) — transfer. horizon 비교는 `18_report_horizon`

In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP)